# Q14 — Qwen1.5-7B: Over-Refusal Direction Analysis

**Mirrors:** `14_[Analysis]_Over_Refusal_Direction_vs_Arditi.ipynb` (LLaMA)

**Key limitation:** Qwen has only 1 refused-harmful sample (RH n=1), so:  
- The harmful-refusal DIM direction cannot be computed  
- `cos(OR direction, global DIM)` is **not available**  
- The harmful-refusal inter-task alignment band cannot be drawn  

**What we CAN compute:**  
- The global OR DIM direction at each layer  
- Per-task OR directions (for tasks with ≥5 OR samples)  
- Pairwise cosine similarity between per-task OR directions  
- OR direction layer profile (magnitude, stability)

**Qwen-specific:** 31 layers, constellation peak at L5.

**No GPU required.**

In [ ]:
EMBEDDINGS_DIR = './embeddings_qwen'
FIGURES_DIR    = './figures'

# Qwen architecture
N_LAYERS_QWEN  = 31
KNOWN_PEAK_L   = 5   # from Q13a
MIN_SAMPLES    = 5   # minimum OR or target samples for per-task direction

import os
os.makedirs(FIGURES_DIR, exist_ok=True)
print('Config set.')

In [ ]:
# COLAB ONLY — uncomment if running on Colab
# from google.colab import drive; drive.mount('/content/drive')
# import os; os.makedirs(EMBEDDINGS_DIR, exist_ok=True)
# !cp -a "/content/drive/MyDrive/embeddings/overalign_eval_qwen_1_5/." {EMBEDDINGS_DIR}/.
print('(Colab cell skipped)')

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from itertools import combinations
import warnings

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.size': 13, 'font.family': 'serif',
    'axes.titlesize': 14, 'axes.titleweight': 'bold',
    'axes.labelsize': 13, 'legend.fontsize': 11,
    'xtick.labelsize': 11, 'ytick.labelsize': 11,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.3,
    'axes.spines.top': False, 'axes.spines.right': False,
    'lines.linewidth': 2.2,
})

TASK_COLORS = {
    'sentiment_analysis': '#E74C3C',
    'translate':          '#3498DB',
    'rephrase':           '#F39C12',
    'rag_qa':             '#27AE60',
    'cryptanalysis':      '#9B59B6',
}
BENIGN_TASKS = ['sentiment_analysis', 'translate', 'cryptanalysis', 'rag_qa']

PAL = {
    'or_global': '#E74C3C',
    'ref':       '#7F8C8D',
    'llama_ref': '#2C3E50',
}

print('✓ Libraries loaded')

In [ ]:
def to_np(d):
    return {
        k: (v.float().numpy().astype(np.float32) if isinstance(v, torch.Tensor)
            else np.array([e.float().numpy().astype(np.float32) for e in v]))
        for k, v in d.items()
    }

csv_files = sorted(f for f in os.listdir(EMBEDDINGS_DIR) if f.endswith('.csv'))
csv_df    = pd.read_csv(os.path.join(EMBEDDINGS_DIR, csv_files[-1]))
torch_path = csv_df['torch_path'].iloc[0]
if not os.path.isabs(torch_path):
    torch_path = os.path.join(EMBEDDINGS_DIR, os.path.basename(torch_path))
torch_data = torch.load(torch_path, map_location='cpu')

embeddings_np        = to_np(torch_data['embeddings'])
texts                = torch_data['texts']
text_type_labels     = np.array(torch_data['text_type_labels'])
intended_task_labels = np.array(torch_data['intended_task_labels'])
response_labels      = csv_df['llm_evaluation'].values
refusal_labels       = csv_df['refusal_class'].values

LAYER_NAMES = sorted(
    [k for k in embeddings_np if 'input_norm' in k],
    key=lambda x: int(x.split('layer_')[1].split('_')[0])
)
LAYER_NUMS = [int(ln.split('layer_')[1].split('_')[0]) for ln in LAYER_NAMES]

print(f'Loaded: {len(texts)} samples | {len(LAYER_NAMES)} layers')

In [ ]:
refusing_mask = (refusal_labels == 'direct_refusal') | (refusal_labels == 'indirect_refusal')
answered_mask = refusal_labels == 'direct_answer'
harmful_mask  = text_type_labels == 'harmful_instruction'
benign_mask   = np.isin(intended_task_labels, BENIGN_TASKS)

OVER_REFUSAL_MASK    = benign_mask  & refusing_mask
REFUSED_HARMFUL_MASK = harmful_mask & refusing_mask
HARMLESS_ANSWERED    = benign_mask  & answered_mask
TARGET_MASK = (
    ((response_labels == 'cautious') | (response_labels == 'not_harmful')) & answered_mask
)

print(f'OR: {OVER_REFUSAL_MASK.sum()}  |  RH: {REFUSED_HARMFUL_MASK.sum()}  |  HA: {HARMLESS_ANSWERED.sum()}  |  TGT: {TARGET_MASK.sum()}')
if REFUSED_HARMFUL_MASK.sum() < 5:
    print('⚠ RH n < 5 — cannot compute harmful-refusal direction or cos(OR, DIM).')
    print('  Proceeding with OR-only analyses.')

print()
print('Per-task OR counts:')
for task in BENIGN_TASKS:
    n_or  = ((intended_task_labels == task) & OVER_REFUSAL_MASK).sum()
    n_tgt = ((intended_task_labels == task) & TARGET_MASK).sum()
    flag  = ' ← direction computable' if n_or >= MIN_SAMPLES and n_tgt >= MIN_SAMPLES else ''
    print(f'  {task:<22}: OR={n_or:>3d}  TGT={n_tgt:>3d}{flag}')

In [ ]:
def dim_direction(emb, mask_a, mask_b):
    d = emb[mask_a].mean(0) - emb[mask_b].mean(0)
    n = np.linalg.norm(d)
    return d / (n + 1e-8)

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

# ── Per-task OR directions at each layer ──────────────────────────────────────
task_or_dirs = {}  # {(task, lname): direction}
VALID_TASKS  = []

for task in BENIGN_TASKS:
    m_or  = OVER_REFUSAL_MASK & (intended_task_labels == task)
    m_tgt = TARGET_MASK       & (intended_task_labels == task)
    if m_or.sum() >= MIN_SAMPLES and m_tgt.sum() >= MIN_SAMPLES:
        VALID_TASKS.append(task)
        for lname in LAYER_NAMES:
            emb = embeddings_np[lname]
            task_or_dirs[(task, lname)] = dim_direction(emb, m_or, m_tgt)

# ── Global OR direction at each layer ─────────────────────────────────────────
global_or_dirs = {}
for lname in LAYER_NAMES:
    if OVER_REFUSAL_MASK.sum() >= 2 and TARGET_MASK.sum() >= 2:
        global_or_dirs[lname] = dim_direction(embeddings_np[lname],
                                               OVER_REFUSAL_MASK, TARGET_MASK)

print(f'Tasks with per-task OR direction: {VALID_TASKS}')
print(f'Global OR direction computed at {len(global_or_dirs)} layers.')

In [ ]:
# ── Figure 1: Global OR direction — cosine with itself across layers ───────────
# Stability analysis: how much does the OR direction change layer to layer?
# Also track OR direction magnitude (unnormalized) as a proxy for signal strength.

or_delta_cos = []  # cosine(OR_dir@L_i, OR_dir@L_{i+1})
or_magnitudes = []  # unnormalized magnitude of OR centroid difference

for i, lname in enumerate(LAYER_NAMES):
    emb = embeddings_np[lname]
    # Magnitude of the DIM vector (before normalisation)
    if OVER_REFUSAL_MASK.sum() >= 2 and TARGET_MASK.sum() >= 2:
        raw_d = emb[OVER_REFUSAL_MASK].mean(0) - emb[TARGET_MASK].mean(0)
        or_magnitudes.append(float(np.linalg.norm(raw_d)))
    else:
        or_magnitudes.append(np.nan)

    if i > 0 and LAYER_NAMES[i-1] in global_or_dirs and lname in global_or_dirs:
        or_delta_cos.append(cosine(global_or_dirs[LAYER_NAMES[i-1]], global_or_dirs[lname]))
    elif i > 0:
        or_delta_cos.append(np.nan)

# ── Pairwise cosine between per-task OR directions (at each layer) ──────────
if len(VALID_TASKS) >= 2:
    pairs = list(combinations(VALID_TASKS, 2))
    pair_sims_by_layer = {p: [] for p in pairs}
    for lname in LAYER_NAMES:
        for t1, t2 in pairs:
            if (t1, lname) in task_or_dirs and (t2, lname) in task_or_dirs:
                pair_sims_by_layer[(t1, t2)].append(
                    cosine(task_or_dirs[(t1, lname)], task_or_dirs[(t2, lname)]))
            else:
                pair_sims_by_layer[(t1, t2)].append(np.nan)

    mean_pairwise = [float(np.nanmean([pair_sims_by_layer[p][i] for p in pairs]))
                     for i in range(len(LAYER_NAMES))]
    HAS_PAIRWISE = True
else:
    HAS_PAIRWISE = False
    print(f'⚠ Only {len(VALID_TASKS)} task(s) with sufficient OR — pairwise cosine not available.')

# ── Figure ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.8))
plt.subplots_adjust(wspace=0.35, top=0.85, bottom=0.12)

PEAK_L = KNOWN_PEAK_L

# Left: OR direction magnitude
ax = axes[0]
ax.plot(LAYER_NUMS, or_magnitudes, 'o-', color=PAL['or_global'], lw=2.2, zorder=3)
ax.axvline(PEAK_L, color=PAL['ref'], lw=1.4, ls='-.', label=f'Constellation peak (L{PEAK_L})')
ax.set_xlabel('Layer'); ax.set_ylabel('L2 magnitude of OR centroid diff')
ax.set_title('Global OR direction magnitude\nacross layers', pad=6)
ax.legend(fontsize=10)

# Right: pairwise cosine between per-task OR directions (or layer-to-layer delta)
ax = axes[1]
if HAS_PAIRWISE:
    ax.plot(LAYER_NUMS, mean_pairwise, 'o-', color=PAL['or_global'], lw=2.2, zorder=3,
            label=f'Mean pairwise cos(OR tasks) n={len(VALID_TASKS)}')
    # LLaMA reference: pairwise OR ~0.54, harmful-refusal ~0.85
    ax.axhspan(0.845, 0.858, alpha=0.15, color='#27AE60',
               label='LLaMA harmful-ref band (0.845–0.858)')
    ax.axhline(0.538, color=PAL['llama_ref'], lw=1.4, ls='--',
               label='LLaMA pairwise OR (0.538)')
    ax.legend(fontsize=9)
else:
    # Layer-to-layer stability instead
    ax.plot(LAYER_NUMS[1:], or_delta_cos, 's-', color=PAL['or_global'], lw=2.2, zorder=3,
            label='cos(OR_dir@L, OR_dir@L+1)')
    ax.legend(fontsize=10)

ax.axvline(PEAK_L, color=PAL['ref'], lw=1.4, ls='-.')
ax.set_xlabel('Layer')
ax.set_ylabel('Cosine similarity')
ax.set_title('Per-task OR direction consistency\nacross layers', pad=6)
ax.set_ylim(-0.1, 1.05)

plt.suptitle('Qwen1.5-7B — Over-Refusal Direction Analysis\n'
             '(Note: harmful-refusal DIM direction unavailable — RH n=1)',
             fontsize=11, fontweight='bold')

plt.savefig(f'{FIGURES_DIR}/q_fig14_or_direction.pdf', bbox_inches='tight', dpi=300)
plt.savefig(f'{FIGURES_DIR}/q_fig14_or_direction.png', bbox_inches='tight', dpi=300)
plt.show()

print('=== FIGURE DATA (OR Direction) ===')
peak_mag_i = int(np.nanargmax(or_magnitudes))
print(f'  OR magnitude peak: {or_magnitudes[peak_mag_i]:.2f} at L{LAYER_NUMS[peak_mag_i]}')
if HAS_PAIRWISE:
    non_nan = [(l, v) for l, v in zip(LAYER_NUMS, mean_pairwise) if not np.isnan(v)]
    if non_nan:
        mid_vals = [(l, v) for l, v in non_nan if 5 <= l <= 20]
        if mid_vals:
            print(f'  Mean pairwise OR cosine (mid-layers L5-L20): {np.mean([v for _,v in mid_vals]):.3f}')
    print(f'  LLaMA reference: pairwise OR=0.538, harmful-refusal=0.845–0.858')

In [ ]:
# ── Figure 2: 2D PCA scatter at peak layer with OR direction arrows ───────────
# Mirrors: fig_nb13b_l12_snapshot (LLaMA)
# Shows all 270 samples at L5 with per-task OR arrows

from sklearn.decomposition import PCA as PCA_sk
from matplotlib.patches import FancyArrowPatch
import matplotlib.patheffects as pe

PEAK_LNAME = f'layer_{PEAK_L}_input_norm'
if PEAK_LNAME not in embeddings_np:
    PEAK_LNAME = LAYER_NAMES[KNOWN_PEAK_L] if KNOWN_PEAK_L < len(LAYER_NAMES) else LAYER_NAMES[0]

emb    = embeddings_np[PEAK_LNAME]
pca2   = PCA_sk(n_components=2, random_state=42)
emb2d  = pca2.fit_transform(emb)
var    = pca2.explained_variance_ratio_

fig, ax = plt.subplots(figsize=(5.5, 4.5))

# Scatter all samples by task
for task in sorted(np.unique(intended_task_labels)):
    m = intended_task_labels == task
    ax.scatter(emb2d[m, 0], emb2d[m, 1], c=TASK_COLORS.get(task, '#95A5A6'),
               s=18, alpha=0.55, edgecolors='white', linewidths=0.3, zorder=2)
    # OR points (triangles)
    m_or = m & OVER_REFUSAL_MASK
    if m_or.any():
        ax.scatter(emb2d[m_or, 0], emb2d[m_or, 1], c=TASK_COLORS.get(task, '#95A5A6'),
                   s=65, alpha=1.0, marker='^', edgecolors='black', linewidths=0.7, zorder=4)

# Per-task OR direction arrows
x_rng = emb2d[:, 0].max() - emb2d[:, 0].min()
y_rng = emb2d[:, 1].max() - emb2d[:, 1].min()
sc    = 0.25 * max(x_rng, y_rng)

def draw_arrow(ax, start, end, color, lw=2.0, zorder=5):
    ax.add_patch(FancyArrowPatch(
        start, end, arrowstyle='->', mutation_scale=12,
        color=color, linewidth=lw, zorder=zorder,
        path_effects=[pe.withStroke(linewidth=lw+2, foreground='white')]
    ))

for task in VALID_TASKS:
    m_or  = OVER_REFUSAL_MASK & (intended_task_labels == task)
    m_tgt = TARGET_MASK       & (intended_task_labels == task)
    if not m_or.any() or not m_tgt.any():
        continue
    ha_c = emb2d[m_tgt].mean(0)
    or_c = emb2d[m_or].mean(0)
    draw_arrow(ax, tuple(ha_c), tuple(or_c), TASK_COLORS[task], lw=2.5)

# Global OR direction arrow (from overall centroid)
if PEAK_LNAME in global_or_dirs:
    cx, cy = emb2d.mean(0)
    d2 = pca2.components_ @ global_or_dirs[PEAK_LNAME]
    d2 = d2 / (np.linalg.norm(d2) + 1e-8)
    draw_arrow(ax, (cx - d2[0]*sc*0.5, cy - d2[1]*sc*0.5),
                   (cx + d2[0]*sc*0.5, cy + d2[1]*sc*0.5),
                   '#1a1a1a', lw=3.5, zorder=8)

# Legend
handles = [mpatches.Patch(color=TASK_COLORS[t], label=t.replace('_',' ')) for t in BENIGN_TASKS]
handles += [
    Line2D([0],[0], marker='o', color='grey', lw=0, ms=5, label='Answered (●)'),
    Line2D([0],[0], marker='^', color='grey', lw=0, ms=7,
           markeredgecolor='k', markeredgewidth=0.7, label='Over-refusal (▲)'),
    Line2D([0],[0], color='#1a1a1a', lw=2.5, label='Global OR direction'),
    Line2D([0],[0], color='grey', lw=2.0, label='Per-task OR arrows (TGT→OR)'),
]
ax.legend(handles=handles, fontsize=8, loc='upper left', framealpha=0.95)

ax.set_xlabel(f'PC1 ({var[0]:.1%})')
ax.set_ylabel(f'PC2 ({var[1]:.1%})')
ax.set_title(f'Qwen1.5-7B — All samples at L{PEAK_L}\nColoured arrows: per-task OR directions')

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/q_fig14_l{PEAK_L}_snapshot.pdf', bbox_inches='tight', dpi=300)
plt.savefig(f'{FIGURES_DIR}/q_fig14_l{PEAK_L}_snapshot.png', bbox_inches='tight', dpi=300)
plt.show()
print(f'✓ Saved: q_fig14_l{PEAK_L}_snapshot')

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
print('=' * 65)
print('Q14 — QWEN OR DIRECTION SUMMARY')
print('=' * 65)
print(f'OR samples: {OVER_REFUSAL_MASK.sum()}')
print(f'RH samples: {REFUSED_HARMFUL_MASK.sum()}  ← too few for DIM direction')
print(f'Tasks with per-task OR direction: {VALID_TASKS}')
print()
print('AVAILABLE:')
print('  ✓ Global OR direction magnitude across layers')
print('  ✓ Per-task OR direction arrows at peak layer')
print('  ✓ Pairwise OR cosine (if ≥2 tasks with sufficient OR)' if HAS_PAIRWISE else '  ✗ Pairwise OR cosine (only 1 valid task)')
print()
print('NOT AVAILABLE (Qwen RH n=1):')
print('  ✗ cos(OR direction, global harmful-refusal DIM)')
print('  ✗ Harmful-refusal inter-task alignment band')
print('  ✗ Refusal-type probe (see Q17)')
print()
print('PAPER IMPLICATION:')
print('  Qwen cannot replicate Claims 2-3 (directional geometry)')
print('  due to near-zero harmful refusal rate on this evaluation set.')
print('  Claim 1 (constellations) replicated — see Q13a.')
print('=' * 65)